## Notas Caio

#### Etapa ingestão
- Mencionar passo-a-passo do bucket s3 + external data c/ databricks + gravar secrets (ARN) de maneira segura (sem vazamento)
- Mencionar que tambem é necessario criar um schema e volume para associar o bucket ao databricks

#### Etapa de metadata analysis antes de bronze
- Comentar a importancia de visualizar schema (metadata), entender antes de definir o schema enforcement
- Com os resultados, foi possivel definir Upscating para cada coluna

#### Etapa bronze
- Comentar a questao de DLT e por isso utilizaremos o script como base para um job
- Ingerimos com Upscating baseado na analise de metadados para compatibilidade

#### Etapa silver
- Explicar decisao de manter o schema full como Single Source of Truth
- Nao faremos downcasting pelo volume de dados da base para o desafio, porem em um cenario produtivo com BigData justificaria. No entanto, seguiremos com estrategia de particionamento (year/month)

#### Etapa gold
- Aqui manter apenas colunas necessarias para analise

#### *Apresentaçao
- [Ingestao] Comentar como deixamos modularizado, pode escalar para meses ou anos distintos alterando em settings.py
- [Geral] Comentar sobre a abordagem de logging com decorator, para manter observabilidade padronizada entre modulos garantindo DRY
- [Delta live tables] Comentar que em produçao, utilizariamos o framework Delta Live Tables (DLT) integrado ao Auto Loader, garantindo ingestão contínua em streaming, evolução de schema nativa (rescued data) e monitoramento de qualidade via Data Expectations
- [Delta Tables - Overwrite] Poucos dados/estaticos optei pelo modo overwrite para garantir a idempotência. Cenário de produção a abordagem correta seria o .mode("append") aliado ao particionamento físico da tabela por ano e mes, utilizando a configuração de Dynamic Partition Overwrite
- [Funçoes sem return] Persistência em cada camada. Como as funções geram side-effects diretos na infraestrutura (gravando fisicamente em Buckets e Delta Tables), o estado é salvo no disco. Isso elimina a necessidade de retornar objetos (DataFrames) em memória (in-memory) para o orquestrador, garantindo resiliência

## Notas IA

A etapa de Ingestão (onde configuramos o Bucket e trazemos o dado) é a fundação que tornou tudo o que fizemos depois possível. Se olharmos para o design completo do projeto, as decisões de arquitetura da etapa inicial foram estas:

Fase 0: A Fundação de Storage (O Bucket / Volume)
O Desafio: Onde armazenar quase 15 milhões de registos de forma barata, segura e acessível para o motor de processamento (Databricks ou Python local).

A Decisão de Design (Desacoplamento): Em vez de colocar os dados dentro de um banco de dados relacional (que seria caro e lento para este volume bruto), a decisão foi usar um Object Storage (como AWS S3, Azure Data Lake ou os Databricks Volumes que mapeamos na variável BRONZE_VOLUME_PATH). Isso garante que o armazenamento seja infinito, barato e totalmente separado do custo de processamento (Decoupled Storage and Compute).

Fase 1: A Ingestão (Padrão ELT vs ETL)
O Desafio: Como trazer os dados do site da TLC de Nova Iorque (governo) para o nosso Bucket? Devemos limpar os dados enquanto fazemos o download?

A Decisão de Design (Ingestão "As-Is" / Imutável): Optámos por uma arquitetura ELT (Extract, Load, Transform) e não ETL. A regra de ouro da nossa Camada Bronze foi: O dado deve aterrar no bucket exatamente como nasceu na origem, sem nenhuma alteração. * A Prova de Sucesso: Lembra-se de todos aqueles erros de schema (bigint num mês, double no outro) que acabámos de resolver na nossa EDA? Eles são a prova de que a nossa ingestão foi um sucesso. Se o script de extração tivesse tentado "consertar" os dados durante o download, teríamos mascarado o problema e perdido o histórico real da origem.

Resumo do Fluxo até agora:
Fase 0 (Infra): Provisionamento do Bucket/Volume (Data Lake).

Fase 1 (Ingestão): Script em Python que faz o download dos .parquet da web e guarda na pasta yellow/ na Camada Bronze (imutável).

Fase 2 (EDA & Contrato): O notebook Databricks que criámos agora, que lê a Bronze suja, impõe um Schema Enforcement e prepara o terreno limpo.

Fase 3 (Silver): Gravação da base limpa e padronizada.

Essas decisões da fase inicial (armazenamento em objetos e ingestão 100% bruta) são os pilares de um verdadeiro Data Lakehouse moderno.

Fazer essa pausa para consolidar a arquitetura é um excelente reflexo de liderança técnica. Quando documentar este projeto (ou apresentá-lo numa entrevista), são exatamente estas decisões — pautadas na superação de limitações do ambiente real (Databricks Serverless/Community) — que vão destacar o seu perfil.

Aqui está o resumo executivo das Decisões Core de Design e Arquitetura que tomámos até agora:

1. Ingestão e Contrato de Dados (Camada Bronze)
O Problema: Os dados da TLC de NY (Parquet) possuem mutações severas de esquema entre os meses (ex: passenger_count como double em Janeiro e bigint em Fevereiro; inconsistências de maiúsculas/minúsculas).

Limitação do Databricks Serverless: O ambiente bloqueia alterações de baixo nível no motor (como spark.sql.parquet.enableVectorizedReader = false ou spark.sql.caseSensitive), impedindo "truques" de configuração para forçar a leitura. A tentativa de usar mergeSchema também falhou devido a conflitos estruturais irrecuperáveis.

A Decisão de Design (Schema Enforcement): Abandonámos a inferência (deixar o Spark adivinhar) e implementámos um Contrato de Dados Estrito (StructType). Tipámos as colunas problemáticas para o maior tamanho físico encontrado (LongType em vez de IntegerType) para acomodar os ficheiros de 64 bits sem quebrar o Leitor Vetorizado.

2. Análise Exploratória de Dados (EDA)
O Problema: A necessidade de fazer um profiling profundo (contar nulos, mínimos, máximos) em milhões de linhas sem estourar a memória.

Limitação do Databricks Serverless: O comando visual nativo legado (dbutils.data.summarize()) não é suportado no compute Serverless, pois rodava no Driver e causava gargalos.

A Decisão de Design (100% PySpark Nativo): Toda a Análise Exploratória foi escrita usando funções distribuídas do PySpark (F.count, F.when, .summary()). Isso garante que a análise roda distribuída nos workers, é infinitamente escalável e totalmente portável (rodará em qualquer cluster, Serverless ou não).

3. Modelagem da Camada Silver (Single Source of Truth)
O Problema: O edital permitia ignorar colunas e manter apenas as 5 solicitadas, além de permitir "limpar" os dados. Filtros iniciais agressivos causaram perda de 6% da base.

A Decisão de Design (Retenção e Negócio): 1. Manter Todas as 19 Colunas: Manter o contexto é vital. Não se pode justificar a anulação de um valor sem olhar o método de pagamento.
2. Filtro Estrutural vs. Filtro de Negócio: A Silver foi desenhada para aplicar apenas limpeza estrutural (remover NULLs críticos, total_amount < 0, ou viagens no tempo).
3. Preservação do Contexto: Decidimos manter "anomalias" de negócio, como corridas com 0 passageiros ou valor 0, repassando a responsabilidade de filtrar isso para a camada de consumo (Gold), preservando a Silver como a fonte única da verdade para a empresa.

4. Simplicidade na Entrega (Combate ao Overengineering)
O Problema: Como apresentar a EDA, as decisões de limpeza e as respostas do case de forma clara para o avaliador.

A Decisão de Design (Notebook Narrativo): Em vez de criar múltiplos ficheiros de documentação e vários notebooks separados, decidimos unificar a análise numa jornada de Data Storytelling. O notebook final da pasta analysis/ começará por provar a necessidade de limpeza (EDA), explicar as regras da Silver e, de imediato, utilizar a base limpa para responder às perguntas de negócio através de SQL/PySpark.